# 🧠 AI 3D Stress Validator — V2: GNN Edition

> **Physics-Informed Structural Validation** using Graph Neural Networks.
>
> Instead of voxelizing meshes into 3D grids, V2 converts STL files into **mesh-graphs** where every vertex is a node and every edge carries structural information. A **Graph Convolutional Network** then "simulates" stress flow through the geometry.

| V1 (Voxel/CNN) | V2 (Graph/GNN) |
|---|---|
| Binary occupancy grid | Vertex positions + surface normals |
| Grid scans empty space | Graph follows real geometry |
| No edge info | Euclidean distance on every edge |
| 3D CNN (Conv3D) | GCN with message passing |

---

**Runtime:** Set to **GPU** → `Runtime → Change runtime type → T4 GPU`

## Phase 0 — Setup & Dependencies

In [ ]:
# 0.1  Install dependencies
# PyTorch Geometric requires matching torch + CUDA versions
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

print(f'PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}')

!pip install -q torch-geometric
!pip install -q trimesh open3d scipy pandas matplotlib

# Verify installation
import torch_geometric
print(f'✅ PyTorch Geometric {torch_geometric.__version__} installed.')

In [ ]:
# 0.2  Configuration constants
import os

# --- Data ---
DATA_ROOT    = '/content/deepjeb_data'    # Where STL files will be stored
REPO_URL     = 'https://github.com/janu3605/AI_3D_Tolerance_Stress_Validator.git'
REPO_DIR     = '/content/AI_3D_Tolerance_Stress_Validator'
NUM_SAMPLES  = 250                         # Number of STL samples to use

# --- Preprocessing ---
TARGET_FACES = 2000                        # QEM decimation target (~80% reduction)

# --- Labels ---
STRESS_FAIL_THRESHOLD = 0.80               # quantile for pass/fail split

# --- GNN Architecture ---
IN_CHANNELS      = 6                       # [x, y, z, nx, ny, nz]
HIDDEN_CHANNELS  = 128                     # latent dimension
NUM_GNN_LAYERS   = 4                       # message-passing depth
EDGE_DIM         = 1                       # [distance]
DROPOUT          = 0.3

# --- Training ---
BATCH_SIZE   = 16
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 80
WEIGHT_DECAY = 1e-4
VAL_SPLIT    = 0.2

# --- Loss weights ---
LAMBDA_STRESS = 1.0   # weight for stress regression loss (MSE)
LAMBDA_RISK   = 0.5   # weight for node-risk classification loss (BCE)

# --- Device ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 0.3  Clone repository (for utils/)
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}')

# Add to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('✅ Repository ready.')

## Phase 1 — Data Loading & Graph Preprocessing

In [ ]:
# 1.1  Download the DeepJEB dataset (same as V1)
# If you already have the data on Google Drive, mount it instead:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   DATA_ROOT = '/content/drive/MyDrive/deepjeb_data'

import glob

# Check if data already exists
stl_files = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.stl'), recursive=True))
if len(stl_files) >= NUM_SAMPLES:
    print(f'✅ Found {len(stl_files)} STL files already in {DATA_ROOT}')
else:
    print('⬇️  Downloading DeepJEB dataset...')
    # Adjust this download command for your data source
    # Option A: From Google Drive
    #   !gdown <file_id> -O /content/deepjeb.zip
    # Option B: From direct URL
    #   !wget <url> -O /content/deepjeb.zip
    # Then unzip:
    #   !unzip -q /content/deepjeb.zip -d {DATA_ROOT}
    print('⚠️  Please download the DeepJEB dataset and place STL files in:', DATA_ROOT)

stl_files = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.stl'), recursive=True))
stl_files = stl_files[:NUM_SAMPLES]
print(f'📁 Using {len(stl_files)} STL samples.')

In [ ]:
# 1.2  Load stress labels from bracket_labels.csv
import pandas as pd
import numpy as np

# Search for the CSV
csv_paths = (
    glob.glob(os.path.join(DATA_ROOT, '**', 'bracket_labels.csv'), recursive=True) +
    glob.glob('/content/drive/MyDrive/**/bracket_labels.csv', recursive=True)
)

if not csv_paths:
    raise FileNotFoundError(
        'bracket_labels.csv not found! Upload it to '
        f'{DATA_ROOT}/ or your Google Drive.'
    )

csv_path = csv_paths[0]
print(f'Loading FEA data from: {csv_path}')
df = pd.read_csv(csv_path)

# Stress columns
STRESS_COLS = [
    'max_ver_stress(MPa)', 'max_hor_stress(MPa)',
    'max_dia_stress(MPa)', 'max_tor_stress(MPa)',
]
missing = [c for c in STRESS_COLS if c not in df.columns]
if missing:
    STRESS_COLS = [c for c in df.columns if 'stress' in c.lower()]
    print(f'Using fallback stress columns: {STRESS_COLS}')

df['max_stress_all'] = df[STRESS_COLS].max(axis=1)
threshold = df['max_stress_all'].quantile(STRESS_FAIL_THRESHOLD)
df['label'] = (df['max_stress_all'] >= threshold).astype(float)

# Build lookups
stress_lookup = {str(row['item_name']): float(row['max_stress_all']) for _, row in df.iterrows()}
label_lookup  = {str(row['item_name']): float(row['label']) for _, row in df.iterrows()}

n_fail = int(df['label'].sum())
print(f'\n📊 Stress threshold (q={STRESS_FAIL_THRESHOLD}): {threshold:.2f} MPa')
print(f'🏷️  {len(df)-n_fail} PASS, {n_fail} FAIL ({100*n_fail/len(df):.1f}% fail rate)')

In [ ]:
# 1.3  Convert STL files to graphs (the key V2 step!)
from utils.mesh_to_graph import mesh_to_graph
import time

print(f'🔄 Converting {len(stl_files)} STL files to graphs...')
print(f'   Target faces after decimation: {TARGET_FACES}')
print(f'   Node features: [x, y, z, nx, ny, nz] (6D)')
print(f'   Edge features: [euclidean_distance] (1D)')
print()

start_time = time.time()
graph_list = []
skipped = 0

for i, stl_path in enumerate(stl_files):
    try:
        # Convert mesh to graph
        data = mesh_to_graph(stl_path, target_faces=TARGET_FACES)

        # Attach labels
        basename = os.path.splitext(os.path.basename(stl_path))[0]
        stress_val = stress_lookup.get(basename, 0.0)
        label = label_lookup.get(basename, 0.0)

        data.y = torch.tensor([label], dtype=torch.float32)
        data.stress_value = torch.tensor([stress_val], dtype=torch.float32)
        data.item_name = basename

        graph_list.append(data)

        if (i + 1) % 25 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            print(f'  ✅ {i+1}/{len(stl_files)} done  '
                  f'({rate:.1f} meshes/sec, '
                  f'last graph: {data.num_nodes} nodes, '
                  f'{data.edge_index.shape[1]} edges)')

    except Exception as e:
        skipped += 1
        print(f'  ⚠️  Skipped {os.path.basename(stl_path)}: {e}')

elapsed = time.time() - start_time
print(f'\n✅ Converted {len(graph_list)} graphs in {elapsed:.1f}s ({skipped} skipped)')

In [ ]:
# 1.4  Inspect a sample graph
sample = graph_list[0]
print('📐 Sample graph structure:')
print(f'   Nodes     : {sample.num_nodes}')
print(f'   Edges     : {sample.edge_index.shape[1]} (bidirectional)')
print(f'   Node feat : {sample.x.shape}  →  [x, y, z, nx, ny, nz]')
print(f'   Edge attr : {sample.edge_attr.shape}  →  [distance]')
print(f'   Label     : {sample.y.item()} ({"FAIL" if sample.y.item() == 1 else "PASS"})')
print(f'   Stress    : {sample.stress_value.item():.2f} MPa')
print(f'   Name      : {sample.item_name}')

In [ ]:
# 1.5  Visualize a sample mesh-graph
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5))

# --- Plot 1: Node positions colored by normal direction ---
ax1 = fig.add_subplot(131, projection='3d')
pos = sample.pos.numpy()
normals = sample.x[:, 3:6].numpy()
# Color by normal z-component (highlights flat vs curved surfaces)
colors = (normals[:, 2] + 1) / 2  # map [-1,1] to [0,1]
sc = ax1.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
                 c=colors, cmap='coolwarm', s=3, alpha=0.7)
ax1.set_title(f'Nodes ({sample.num_nodes})', fontsize=10)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# --- Plot 2: Edge connections (subset for clarity) ---
ax2 = fig.add_subplot(132, projection='3d')
edge_idx = sample.edge_index.numpy()
# Plot every Nth edge to avoid visual clutter
step = max(1, edge_idx.shape[1] // 500)
for j in range(0, edge_idx.shape[1], step):
    src, dst = edge_idx[0, j], edge_idx[1, j]
    pts = pos[[src, dst]]
    ax2.plot(pts[:, 0], pts[:, 1], pts[:, 2],
             'b-', alpha=0.15, linewidth=0.3)
ax2.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c='red', s=1, alpha=0.5)
ax2.set_title(f'Edges ({edge_idx.shape[1]})', fontsize=10)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

# --- Plot 3: Edge length distribution ---
ax3 = fig.add_subplot(133)
edge_lengths = sample.edge_attr.numpy().flatten()
ax3.hist(edge_lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax3.set_title('Edge Length Distribution', fontsize=10)
ax3.set_xlabel('Normalized Distance')
ax3.set_ylabel('Count')

fig.suptitle(f'Graph: {sample.item_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 2 — GNN Training

In [ ]:
# 2.1  Train / Validation split + DataLoaders
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

# Stratified split on labels
labels = [int(d.y.item()) for d in graph_list]
train_data, val_data = train_test_split(
    graph_list, test_size=VAL_SPLIT,
    stratify=labels, random_state=42
)

print(f'📦 Train: {len(train_data)} graphs')
print(f'📦 Val  : {len(val_data)} graphs')

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)

# Quick sanity check
batch = next(iter(train_loader))
print(f'\n🔍 Sample batch:')
print(f'   Batch x     : {batch.x.shape}')
print(f'   Batch edges  : {batch.edge_index.shape}')
print(f'   Batch labels : {batch.y.shape}')
print(f'   Batch vector : {batch.batch.shape} (max={batch.batch.max().item()})')

In [ ]:
# 2.2  Define GNN model, optimizer, loss functions
from utils.gnn_model import StressGNN

model = StressGNN(
    in_channels=IN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    num_gnn_layers=NUM_GNN_LAYERS,
    edge_dim=EDGE_DIM,
    dropout=DROPOUT,
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🧠 StressGNN Model')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')
print(f'   GNN layers      : {NUM_GNN_LAYERS}')
print(f'   Hidden dim      : {HIDDEN_CHANNELS}')
print()
print(model)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler (reduce on plateau)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

# Loss functions
stress_loss_fn = nn.MSELoss()          # Regression: predict max stress
risk_loss_fn   = nn.BCELoss()          # Per-node: risk classification

print(f'\n⚙️  Optimizer: Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'   Loss = {LAMBDA_STRESS}×MSE(stress) + {LAMBDA_RISK}×BCE(node_risk)')

In [ ]:
# 2.3  Training loop
import time

history = {
    'train_loss': [], 'val_loss': [],
    'train_stress_mae': [], 'val_stress_mae': [],
}

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

print(f'🚀 Training for {NUM_EPOCHS} epochs on {DEVICE}...')
print(f'   Early stopping patience: {EARLY_STOP_PATIENCE}')
print('=' * 70)

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    train_loss_sum = 0.0
    train_mae_sum = 0.0
    train_count = 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        out = model(batch)

        # Stress regression loss
        stress_pred = out['stress']              # (B, 1)
        stress_true = batch.stress_value.view(-1, 1)  # (B, 1)
        loss_stress = stress_loss_fn(stress_pred, stress_true)

        # Node-risk classification loss
        # Use the graph-level label as a soft target for all nodes in that graph
        node_risk = out['node_risk']             # (N_total, 1)
        node_labels = batch.y[batch.batch].view(-1, 1)  # broadcast graph label to nodes
        loss_risk = risk_loss_fn(node_risk, node_labels)

        # Combined loss
        loss = LAMBDA_STRESS * loss_stress + LAMBDA_RISK * loss_risk
        loss.backward()
        optimizer.step()

        bs = stress_true.shape[0]
        train_loss_sum += loss.item() * bs
        train_mae_sum += (stress_pred - stress_true).abs().sum().item()
        train_count += bs

    train_loss = train_loss_sum / train_count
    train_mae = train_mae_sum / train_count

    # ---- VALIDATE ----
    model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            out = model(batch)

            stress_pred = out['stress']
            stress_true = batch.stress_value.view(-1, 1)
            loss_stress = stress_loss_fn(stress_pred, stress_true)

            node_risk = out['node_risk']
            node_labels = batch.y[batch.batch].view(-1, 1)
            loss_risk = risk_loss_fn(node_risk, node_labels)

            loss = LAMBDA_STRESS * loss_stress + LAMBDA_RISK * loss_risk

            bs = stress_true.shape[0]
            val_loss_sum += loss.item() * bs
            val_mae_sum += (stress_pred - stress_true).abs().sum().item()
            val_count += bs

    val_loss = val_loss_sum / val_count
    val_mae = val_mae_sum / val_count

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_stress_mae'].append(train_mae)
    history['val_stress_mae'].append(val_mae)

    # LR scheduler step
    scheduler.step(val_loss)

    # Best model checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), 'best_gnn_model.pth')
    else:
        patience_counter += 1

    # Print progress
    if epoch % 5 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} │ '
              f'Train Loss: {train_loss:.4f}  MAE: {train_mae:.2f} │ '
              f'Val Loss: {val_loss:.4f}  MAE: {val_mae:.2f} │ '
              f'LR: {lr:.1e} │ '
              f'Best: ep{best_epoch}')

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\n⏹️  Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)')
        break

train_time = time.time() - train_start
print(f'\n✅ Training complete in {train_time/60:.1f} min')
print(f'   Best val loss: {best_val_loss:.4f} at epoch {best_epoch}')

In [ ]:
# 2.4  Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#F44336')
axes[0].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Combined Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE curves
axes[1].plot(history['train_stress_mae'], label='Train MAE', color='#2196F3')
axes[1].plot(history['val_stress_mae'],   label='Val MAE',   color='#F44336')
axes[1].axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Stress MAE (MPa)')
axes[1].set_title('Stress Prediction MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('GNN Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 3 — Evaluation & Inference

In [ ]:
# 3.1  Evaluate on validation set
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score

# Load best model
model.load_state_dict(torch.load('best_gnn_model.pth', map_location=DEVICE, weights_only=True))
model.eval()

all_stress_pred = []
all_stress_true = []
all_label_pred  = []
all_label_true  = []

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(DEVICE)
        out = model(batch)

        # Stress regression
        stress_pred = out['stress'].cpu().numpy().flatten()
        stress_true = batch.stress_value.cpu().numpy().flatten()
        all_stress_pred.extend(stress_pred)
        all_stress_true.extend(stress_true)

        # Classification (threshold: if predicted stress >= threshold → FAIL)
        label_true = batch.y.cpu().numpy().flatten()
        label_pred = (stress_pred >= threshold).astype(float)
        all_label_pred.extend(label_pred)
        all_label_true.extend(label_true)

all_stress_pred = np.array(all_stress_pred)
all_stress_true = np.array(all_stress_true)

# Metrics
mae = mean_absolute_error(all_stress_true, all_stress_pred)
r2  = r2_score(all_stress_true, all_stress_pred)
acc = accuracy_score(all_label_true, all_label_pred)

print('=' * 50)
print('📊  Validation Results (Best Model)')
print('=' * 50)
print(f'  Stress MAE      : {mae:.2f} MPa')
print(f'  Stress R²       : {r2:.4f}')
print(f'  Classification  : {acc*100:.1f}% accuracy')
print(f'  Samples         : {len(all_stress_true)}')
print('=' * 50)

# Scatter plot: predicted vs true stress
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(all_stress_true, all_stress_pred, alpha=0.6, c='steelblue', s=40)
lims = [min(all_stress_true.min(), all_stress_pred.min()),
        max(all_stress_true.max(), all_stress_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('True Stress (MPa)', fontsize=12)
ax.set_ylabel('Predicted Stress (MPa)', fontsize=12)
ax.set_title(f'GNN Stress Prediction (R² = {r2:.3f})', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3.2  Single-file inference
from utils.mesh_to_graph import mesh_to_graph
from utils.gnn_model import predict_stress

def analyze_stl(stl_path: str):
    """
    Run the full V2 inference pipeline on a single STL file.

    Returns dict with predicted stress, risk score, and risk location.
    """
    print(f'\n🔍 Analyzing: {os.path.basename(stl_path)}')
    print('   Step 1: Converting mesh to graph...')
    data = mesh_to_graph(stl_path, target_faces=TARGET_FACES)
    print(f'   → {data.num_nodes} nodes, {data.edge_index.shape[1]} edges')

    print('   Step 2: Running GNN inference...')
    result = predict_stress(model, data, device=DEVICE)

    stress = result['predicted_stress']
    risk = result['max_risk_score']
    risk_pos = result['max_risk_position']
    status = '🔴 FAIL' if stress >= threshold else '🟢 PASS'

    print(f'\n   ═══════════════════════════════════════')
    print(f'   Result: {status}')
    print(f'   Predicted Max Stress : {stress:.2f} MPa')
    print(f'   Fail Threshold       : {threshold:.2f} MPa')
    print(f'   Max Risk Score       : {risk:.4f}')
    print(f'   Risk Zone Center     : ({risk_pos[0]:.2f}, {risk_pos[1]:.2f}, {risk_pos[2]:.2f})')
    print(f'   ═══════════════════════════════════════')

    return result, data

# --- Run on a validation sample ---
sample_path = stl_files[0]  # Change index or use your own STL file
result, data = analyze_stl(sample_path)

In [ ]:
# 3.3  Visualization — 3D risk heatmap + danger zone bounding box
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def visualize_risk(
    data,
    result,
    stl_name: str = 'Part',
    bbox_radius: float = None,
):
    """
    3D scatter plot of the mesh colored by per-node risk score,
    with a translucent red bounding box around the danger zone.
    """
    # Get per-node risk scores
    model.eval()
    data_device = data.to(DEVICE)
    with torch.no_grad():
        out = model(data_device)
    node_risk = out['node_risk'].cpu().numpy().flatten()

    pos = data.pos.numpy()
    risk_center = result['max_risk_position']

    # Auto-compute bbox radius if not specified
    if bbox_radius is None:
        extents = pos.max(axis=0) - pos.min(axis=0)
        bbox_radius = extents.max() * 0.08  # 8% of largest extent

    fig = plt.figure(figsize=(16, 7))

    # --- Plot 1: Full model with risk heatmap ---
    ax1 = fig.add_subplot(121, projection='3d')
    sc = ax1.scatter(
        pos[:, 0], pos[:, 1], pos[:, 2],
        c=node_risk, cmap='YlOrRd', s=5, alpha=0.8,
        vmin=0, vmax=1,
    )
    plt.colorbar(sc, ax=ax1, shrink=0.6, label='Risk Score')
    ax1.set_title(f'{stl_name} — Node Risk Heatmap', fontsize=11)
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

    # --- Plot 2: Danger zone close-up with bounding box ---
    ax2 = fig.add_subplot(122, projection='3d')

    # Color points by risk
    sc2 = ax2.scatter(
        pos[:, 0], pos[:, 1], pos[:, 2],
        c=node_risk, cmap='YlOrRd', s=5, alpha=0.6,
        vmin=0, vmax=1,
    )

    # Draw translucent red bounding box around danger zone
    r = bbox_radius
    cx, cy, cz = risk_center

    # 8 corners of the bounding box
    corners = np.array([
        [cx-r, cy-r, cz-r], [cx+r, cy-r, cz-r],
        [cx+r, cy+r, cz-r], [cx-r, cy+r, cz-r],
        [cx-r, cy-r, cz+r], [cx+r, cy-r, cz+r],
        [cx+r, cy+r, cz+r], [cx-r, cy+r, cz+r],
    ])

    # 6 faces of the box
    faces = [
        [corners[j] for j in [0, 1, 2, 3]],
        [corners[j] for j in [4, 5, 6, 7]],
        [corners[j] for j in [0, 1, 5, 4]],
        [corners[j] for j in [2, 3, 7, 6]],
        [corners[j] for j in [1, 2, 6, 5]],
        [corners[j] for j in [0, 3, 7, 4]],
    ]

    box = Poly3DCollection(faces, alpha=0.15, facecolor='red', edgecolor='red', linewidth=1.5)
    ax2.add_collection3d(box)

    # Mark the risk center
    ax2.scatter(*risk_center, c='red', s=100, marker='X', zorder=10,
                label=f'Risk Center ({cx:.1f}, {cy:.1f}, {cz:.1f})')

    # Zoom in around the danger zone
    zoom = bbox_radius * 5
    ax2.set_xlim(cx - zoom, cx + zoom)
    ax2.set_ylim(cy - zoom, cy + zoom)
    ax2.set_zlim(cz - zoom, cz + zoom)

    stress_val = result['predicted_stress']
    ax2.set_title(
        f'Danger Zone — {stress_val:.1f} MPa\n'
        f'({"FAIL" if stress_val >= threshold else "PASS"})',
        fontsize=11, color='red' if stress_val >= threshold else 'green'
    )
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    ax2.legend(fontsize=9)

    plt.suptitle(
        f'🧠 GNN Structural Analysis — {stl_name}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

# --- Visualize the sample ---
visualize_risk(data, result, stl_name=os.path.basename(sample_path))

In [ ]:
# 3.4  Upload your own STL file for analysis
try:
    from google.colab import files
    print('📂 Upload an STL file for analysis:')
    uploaded = files.upload()

    for filename in uploaded:
        upload_path = f'/content/{filename}'
        with open(upload_path, 'wb') as f:
            f.write(uploaded[filename])

        result, data = analyze_stl(upload_path)
        visualize_risk(data, result, stl_name=filename)

except ImportError:
    print('Not running in Colab. To analyze a local file:')
    print('  result, data = analyze_stl("/path/to/your/file.stl")')
    print('  visualize_risk(data, result, stl_name="your_file.stl")')

In [ ]:
# 3.5  Save model for deployment
save_path = 'stress_gnn_v2_final.pth'
torch.save(model.state_dict(), save_path)
print(f'💾 Model saved to: {save_path}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Save model config for reproducibility
import json
config = {
    'model': 'StressGNN',
    'version': 'v2',
    'in_channels': IN_CHANNELS,
    'hidden_channels': HIDDEN_CHANNELS,
    'num_gnn_layers': NUM_GNN_LAYERS,
    'edge_dim': EDGE_DIM,
    'target_faces': TARGET_FACES,
    'stress_threshold_mpa': float(threshold),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
}
with open('gnn_model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f'📄 Config saved to: gnn_model_config.json')
print(json.dumps(config, indent=2))